In [ ]:
from util import *
import os
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm



In [ ]:
chunksize = 10**7  # Read in chunks of 10 million rows
df_list = []  # List to store sampled chunks

for num_of_chunk, chunk in enumerate(pd.read_csv("./avazu-ctr-prediction/train.gz", chunksize=chunksize), start=1):
    df_list.append(chunk.sample(frac=1, random_state=42))  # Sample 10%
    print(f'Chunk {num_of_chunk} processed.')

# Concatenate all sampled chunks at once
df = pd.concat(df_list, ignore_index=True)

# Cleanup
del df_list
gc.collect()

In [ ]:


# Define parameters
ATTRIBUTE_LIST = ['C18']
EPS_LIST = [12]
RESULTS_DIR = './results'

# Create the 'results' folder if it doesn't exist
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved in: {RESULTS_DIR}")

# Process each attribute
for attribute in ATTRIBUTE_LIST:
    category_list, individual_distributions_list, _ = process_click_data(df=df, column_name=attribute, row_col='device_id')

    for eps in EPS_LIST:
        results = []
        print(f"\nProcessing attribute: {attribute} | Eps: {eps}")

        for i, distributions_cur in enumerate(individual_distributions_list):
            q = np.array(distributions_cur).mean(axis=0)  # Calculate distribution mean
            sorted_indices = np.argsort(q)
            q = q[sorted_indices]

            if len(q) <= 1:
                continue  # Skip if there's only one unique value

            K = K_with_prior(q, eps)  # Compute K with prior

            # Initialize tracking variables
            min_max_max = husain_max = 0

            # Process each distribution
            for dist in tqdm(distributions_cur, desc=f"Processing {attribute} (eps={eps})"):
                p = np.array(dist)[sorted_indices]

                # Perform projection
                min_max, husain = perform_projection(p, q, eps, K, 'tv')

                # Update max values
                min_max_max = max(min_max_max, min_max)
                husain_max = max(husain_max, husain)

            # Store results
            results.append({
                'Column': category_list[i],
                'Eps': eps,
                'Min-Max Max': min_max_max,
                'Husain Max': husain_max
            })
            print(f"  → Processed Column {category_list[i]}: Min-Max Max={min_max_max}, Husain Max={husain_max}")

        # Convert results to DataFrame
        results_df = pd.DataFrame(results)
        print("\nFinal results DataFrame:")
        print(results_df.round(2))

        # Save results
        output_file = f"{RESULTS_DIR}/df_{eps}_{attribute}.csv"
        results_df.to_csv(output_file, index=False)
        print(f"✔ Results saved: {output_file}")

        # Cleanup
        del results_df, results
        gc.collect()

print("\n✅ Processing complete! All results saved.")
